# Depth-3 latent probes: final three-panel figure

Use checkpoints **512, 16384, 262144, and 5931648** from the 3-layer, width-768, 16-head run (`bin2_ter16`, `l20.0`). The panels show second-last level-1, third-last level-1, and third-last level-2 latents, matching the original three-panel figure.

The experiment preserves the original grammar, independent fully ternary train/test sets (22,000 / 10,000 examples; seeds 42 / 43), causal positions, and standardized logistic probes (`C=0.1`). It uses the original depth-6 bundle only for these experiment definitions; every plotted curve is fitted using the depth-3 models.

## 1. Compute missing depth-3 checkpoints

Run the next cell from the repository root. It saves `second_third_last_probe_checkpoint_results_depth3.pkl` after each completed checkpoint and reuses completed stages. The original depth-6 results and plots are preserved.


In [ ]:
# Run this cell once; completed checkpoints are reused on subsequent runs.
from pathlib import Path
import runpy
import sys

scripts_dir = Path.cwd() / 'scripts'
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))
runpy.run_path(str(scripts_dir / 'probe_latent_depth3.py'), run_name='__main__')


## 2. Regenerate the final three-panel plot

Once all four checkpoints are saved, run the next cell independently, including after restarting the kernel. It needs only NumPy, Matplotlib, and the depth-3 result bundle. It exports `latent_probe_three_panel_summary_depth3.png` and `.pdf` with embedding plus L1–L3 on the horizontal axis.


In [ ]:
# Standalone: run only this cell after restarting the kernel.
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt

# Set DATA_DIR explicitly if the kernel starts outside the notebook directory.
DATA_DIR = Path.cwd()
RESULTS_PATH = DATA_DIR / "second_third_last_probe_checkpoint_results_depth3.pkl"
with RESULTS_PATH.open("rb") as handle:
    plot_data = pickle.load(handle)

final_results = plot_data["results"]
final_layer_labels = plot_data["layer_labels"]
chance_accuracy = plot_data["chance_accuracy"]
print(f"Loaded plot data from {RESULTS_PATH}")

STAGE_STEPS = [512, 16_384, 262_144, 5_931_648]
STEP_COLORS = {
    512: "tab:blue",
    16_384: "tab:orange",
    262_144: "tab:green",
    5_931_648: "tab:red",
}
# Paper notation uses root-based depth and one-based node indices:
# level-1 node 7 -> mu^(2)_8; level-1 node 6 -> mu^(2)_7;
# level-2 node 0 -> mu^(1)_1.
PANELS = [
    ("second_last_level1", r"(a) $\mu^{(2)}_8$"),
    ("third_last_level1", r"(b) $\mu^{(2)}_7$"),
    ("third_last_level2", r"(c) $\mu^{(1)}_1$"),
]

if plot_data['architecture']['num_layers'] != 3 or plot_data['architecture']['embedding_dim'] != 768:
    raise ValueError('Expected the depth-3, width-768 probe bundle')
if len(final_layer_labels) != 4:
    raise ValueError('Expected embedding plus 3 layers')
for step in STAGE_STEPS:
    for name, _ in PANELS:
        curve = np.asarray(final_results[step]['probes'][name]['test'])
        if curve.shape != (4,) or not np.all(np.isfinite(curve)):
            raise ValueError(f'Invalid saved curve: checkpoint {step}, probe {name}')

x_depth = np.arange(len(final_layer_labels))
fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.2), sharex=True, sharey=True)

for ax, (probe_name, title) in zip(axes, PANELS):
    for stage, step in enumerate(STAGE_STEPS, start=1):
        accuracy = final_results[step]["probes"][probe_name]["test"]
        ax.plot(
            x_depth, accuracy, "o-", color=STEP_COLORS[step],
            linewidth=2.0, markersize=5, label=f"Stage {stage}",
        )
    ax.axhline(chance_accuracy, color="gray", linestyle=":", linewidth=1.3)
    ax.set(
        xticks=x_depth, xticklabels=final_layer_labels,
        xlabel="Probed layer", ylim=(0, 1.02),
    )
    ax.set_title(title, fontsize=22, pad=12)
    ax.xaxis.label.set_size(16)
    ax.tick_params(axis="y", labelsize=13)
    ax.tick_params(axis="x", labelsize=11, labelrotation=45)
    ax.grid(alpha=0.2)

axes[0].set_ylabel("Held-out probe accuracy", fontsize=16)
axes[0].legend(loc="lower right", frameon=False, fontsize=13)
fig.tight_layout(w_pad=1.6)

PDF_PATH = DATA_DIR / "latent_probe_three_panel_summary_depth3.pdf"
PNG_PATH = DATA_DIR / "latent_probe_three_panel_summary_depth3.png"
fig.savefig(PDF_PATH, bbox_inches="tight")
fig.savefig(PNG_PATH, dpi=300, bbox_inches="tight")
print(f"Saved figure to {PDF_PATH}")
print(f"Saved figure to {PNG_PATH}")
plt.show()
